In [2]:
import pandas as pd 
import numpy as np
import os

In [95]:
# Before running this cell , make sure you run createFolder.py and you run this file in this dataCleaning folder, which is inside notebook folder
with open("dataFolder.txt","r") as f:
    folder = f.read()

In [39]:
filepath = os.path.join(folder,"02_nav_history.csv")
df = pd.read_csv(filepath)

In [40]:
df.head()

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [41]:
df.shape


(46000, 3)

## Parse dates to date and time

In [42]:
df['date'] = pd.to_datetime(df['date'])

In [43]:
df.dtypes

amfi_code             int64
date         datetime64[ns]
nav                 float64
dtype: object

## Sort by amfi_code and date

In [44]:
df = df.sort_values(by=['amfi_code','date'])

In [48]:
df.head()

,amfi_code,date,nav
5750,100016,2022-01-03,520.4608
5751,100016,2022-01-04,515.0971
5752,100016,2022-01-05,521.7239
5753,100016,2022-01-06,515.7880
5754,100016,2022-01-07,515.1639


In [59]:
df = df.reset_index().drop('index',axis=1)

In [61]:
df.drop_duplicates(inplace=True)

In [62]:
df.head(20)

,amfi_code,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239
3,100016,2022-01-06,515.7880
4,100016,2022-01-07,515.1639
5,100016,2022-01-10,510.7136
6,100016,2022-01-11,513.5542
7,100016,2022-01-12,512.3195
8,100016,2022-01-13,510.2445
9,100016,2022-01-14,514.3636


##  forward-fill missing NAV for holidays/weekends

In [72]:
clean_df = (
    df.set_index('date')
      .groupby('amfi_code')
      .apply(
          lambda g: g.reindex(
              pd.date_range(g.index.min(), g.index.max(), freq='D')
          ).ffill()
      )
      .drop(columns='amfi_code')
      .reset_index()
)

C:\Users\dell\AppData\Local\Temp\ipykernel_16908\3544470538.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [83]:
clean_df.columns = ['amfi_code','date','nav']

In [91]:
clean_df.shape

(64320, 3)

In [94]:
clean_df.drop_duplicates(inplace=True)

## Validate all nav>0 

In [93]:
clean_df[clean_df['nav']<=0]

,amfi_code,date,nav


In [96]:
clean_df

,amfi_code,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239
3,100016,2022-01-06,515.7880
4,100016,2022-01-07,515.1639
...,...,...,...
64315,149324,2026-05-25,292.4810
64316,149324,2026-05-26,291.2707
64317,149324,2026-05-27,288.8007
64318,149324,2026-05-28,280.6873


In [97]:
df

,amfi_code,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239
3,100016,2022-01-06,515.7880
4,100016,2022-01-07,515.1639
...,...,...,...
45995,149324,2026-05-25,292.4810
45996,149324,2026-05-26,291.2707
45997,149324,2026-05-27,288.8007
45998,149324,2026-05-28,280.6873


In [100]:
fol = os.path.dirname(folder)
savePath = os.path.join(fol,"processed")
clean_df.to_csv(os.path.join(savePath,"c_nav_history.csv"),index=False)